In [0]:
%run ./00_config

In [0]:
%run ./02_core_functions

# 03_main_hsar (daily)
For every active **HSAR** row in the control table this notebook:

1. **Copies** new zips from the source into `<zip_tgt_path>/<mid_date>/`
2. **Unzips** them into `<unzip_tgt_path>/<mid_date>/<zip_name>/`
3. **Loads** each affected `<mid_date>` folder with Auto Loader. Each JSON file becomes one row with its full content, tagged with the zip it came from.

Just run it. It checks the whole source folder and only handles zips that aren't in `raw/` yet. The first run loads everything, and after that it only picks up what's new, so a missed day gets caught up by itself.

The preview below warns you if today's zip hasn't arrived.

If a zip fails to unzip, or its folder fails to load, the raw copy is deleted (and the folder's partial rows and checkpoint are cleared), so the next run tries again from scratch.

In [0]:
run_id          = str(uuid.uuid4())
control_entries = get_active_control_entries("HSAR")
ctl_by_sno      = {c["s_no"]: c for c in control_entries}
log_records     = []

print(f"run_id = {run_id}")
print(f"{len(control_entries)} active HSAR control row(s)")

### Preview: Available Files / New Files to load

In [0]:
today = datetime.now().strftime("%Y%m%d")
for ctl in control_entries:
    all_zips = list_zip_files(ctl["zip_source_path"])
    zips     = select_zips_to_copy(all_zips, ctl)
    print(f"s_no={ctl['s_no']}: {len(zips)} new zip(s); END dates to load: {zip_end_dates(zips) or '-'}")
    if today not in zip_end_dates(all_zips):
        print(f"  WARNING: no zip with END date {today} (today) at the source yet")

### Steps 1 and 2: copy and unzip

In [0]:
copy_and_unzip(run_id, control_entries, log_records)

### Step 3: load the dated folders

In [0]:
# which dated folders got zips in this run: {(s_no, yyyymmdd): [unzip records]}
folders = {}
for r in log_records:
    if r["stage"] == "UNZIP" and r["status"] == "SUCCESS":
        folders.setdefault((r["s_no"], zip_folder_date(r["file_name"])), []).append(r)
print(f"{len(folders)} dated folder(s) to load")

load_records = []
for (s_no, dated_folder), unzip_recs in sorted(folders.items()):
    ctl    = ctl_by_sno[s_no]
    target = f"{ctl['tgt_catalog']}.{ctl['tgt_schema']}.{ctl['tgt_table']}"
    rec = {"s_no": s_no, "stage": "HSAR_LOAD", "source": ctl["src_system"], "file_name": dated_folder,
           "source_path": f"{ctl['unzip_tgt_path'].rstrip('/')}/{dated_folder}/", "target_path": target,
           "start_time": datetime.now()}
    try:
        n_files = hsar_load_folder(ctl, dated_folder)
        rec.update(status="SUCCESS", message=f"folder has {n_files} file(s) in the target after loading {len(unzip_recs)} new zip(s)")
    except Exception as e:
        rec.update(status="FAILED", message=str(e)[:4000])
        try:
            hsar_reset_folder(ctl, dated_folder)                  # start clean for the retry
        except Exception as reset_err:
            print(f"WARNING: could not reset {dated_folder}: {reset_err}")
        for u in unzip_recs:
            discard_raw_copy(u["source_path"])                    # so the zip is copied, unzipped and loaded again next run
    rec["end_time"] = datetime.now()
    load_records.append(rec)
    print(f"  [HSAR:{rec['status']}] {dated_folder}: {rec['message'][:300]}")

write_logs(load_records, run_id)
print(f"Logged {len(load_records)} HSAR_LOAD record(s) to {log_table}")

### Run summary

In [0]:
display(spark.table(log_table).filter(F.col("run_id") == run_id).orderBy("start_time"))

In [0]:
%sql
select * from iics.casemix.hsar_json_test
-- where zip_file='carereport_hsar_1007320371375345779_20250309_20250328_20261022_000002.zip'
order by file_id desc

In [0]:
# if anything failed, fail the job too (the details are in the log table under this run_id)
failed = sum(1 for r in log_records + load_records if r["status"] == "FAILED")
if failed > 0:
    raise Exception(f"{failed} step(s) failed. See {log_table} where run_id = '{run_id}'")